# Tutorial: GIS siting workflow

Audience:
- Analysts who need a notebook for site screening and ranked comparison.

Prerequisites:
- `luminus-py` installed with GIS extras if you want GeoDataFrame output.
- A working `luminus-mcp` install on `PATH`.
- Access to the GIS profile used by your environment.

Learning goals:
- Screen a single candidate site.
- Pull grid connection intelligence for a pilot site.
- Inspect nearby SSEN DNO headroom matches where public coverage exists.
- Rank a shortlist of sites in one call.
- Export the ranking table as GeoJSON or GeoDataFrame output.


## Outline

1. Start a GIS profile client.
2. Define a small candidate set.
3. Screen one site for a fast sanity check.
4. Pull transmission and DNO connection context.
5. Rank the whole shortlist.
6. Export map-friendly formats.
7. Capture the next iteration.


In [ ]:
from __future__ import annotations

import pandas as pd
from IPython.display import display
from luminus import (
    DistributionHeadroomSnapshot,
    GridConnectionIntelligenceSnapshot,
    Luminus,
)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

lum = Luminus(profile="gis", request_timeout=120.0)
lum


## Step 1 - Define candidate sites

Keep the input list compact so the notebook reads like a screening memo instead of a bulk data dump. This example stays in Southern England so the SSEN public DNO headroom helper has a chance to resolve nearby context.


In [ ]:
candidate_sites = pd.DataFrame(
    [
        {"label": "Alpha", "lat": 50.840, "lon": -1.080},
        {"label": "Bravo", "lat": 50.875, "lon": -1.145},
        {"label": "Charlie", "lat": 50.915, "lon": -1.025},
        {"label": "Delta", "lat": 50.790, "lon": -1.170},
    ]
)

display(candidate_sites)
candidate_sites.to_dict(orient="records")


## Step 2 - Screen one site

Use the single-site workflow first. It is a quick way to confirm that the profile and coordinates are wired correctly before you run the full shortlist.


In [ ]:
pilot = candidate_sites.iloc[0]
pilot_site = lum.screen_site(lat=float(pilot["lat"]), lon=float(pilot["lon"]), country="GB")
pilot_site.to_dict()


## Step 3 - Pull connection context with the Python GIS helpers

Use the typed snapshot helper when you want a compact narrative summary of transmission queue, nearby substations, and the nearest public SSEN DNO headroom signal.


In [ ]:
intelligence: GridConnectionIntelligenceSnapshot = lum.get_grid_connection_intelligence_snapshot(
    lat=float(pilot["lat"]),
    lon=float(pilot["lon"]),
    country="GB",
)

intelligence_summary = pd.DataFrame(
    [
        {
            "candidate": pilot["label"],
            "gsp_region": intelligence.nearest_gsp.region_name if intelligence.nearest_gsp else None,
            "queue_total_mw": intelligence.connection_queue.total_mw_queued if intelligence.connection_queue else None,
            "headroom_site": intelligence.distribution_headroom.substation if intelligence.distribution_headroom else None,
            "generation_headroom_mw": (
                intelligence.distribution_headroom.estimated_generation_headroom_mw
                if intelligence.distribution_headroom
                else None
            ),
        }
    ]
)

nearby_substations = pd.DataFrame(
    [
        {"name": sub.name, "voltage_kv": sub.voltage_kv, "distance_km": sub.distance_km}
        for sub in intelligence.nearby_substations
    ]
)

display(intelligence_summary)
display(nearby_substations.head())
intelligence.confidence_notes


## Step 4 - Inspect nearby SSEN distribution headroom matches

Use the DataFrame helper when you want a notebook table for filtering, sorting, or handoff into a screening memo. The typed snapshot stays useful for the top-level summary and caveats.


In [ ]:
headroom_snapshot: DistributionHeadroomSnapshot = lum.get_distribution_headroom_snapshot(
    lat=float(pilot["lat"]),
    lon=float(pilot["lon"]),
    operator="SSEN",
    radius_km=25,
)

headroom_matches = lum.get_distribution_headroom_matches(
    lat=float(pilot["lat"]),
    lon=float(pilot["lon"]),
    operator="SSEN",
    radius_km=25,
)

display(
    headroom_matches[
        [
            "substation",
            "distance_km",
            "estimated_generation_headroom_mw",
            "generation_rag_status",
            "estimated_demand_headroom_mva",
            "demand_rag_status",
        ]
    ].head()
)

headroom_snapshot.confidence_notes


## Step 5 - Rank the shortlist

The `compare_sites` result keeps the full payload. Pull the `rankings` table into pandas for the analyst view.


In [ ]:
comparison = lum.compare_sites(country="GB", sites=candidate_sites.to_dict(orient="records"))
rankings = comparison.to_pandas(data_key="rankings")

display(rankings[["rank", "label", "verdict", "nearest_grid_km", "constraint_count", "score"]].head())
rankings.columns.tolist()


## Step 6 - Export mapping outputs

GeoJSON is useful for lightweight review tools. GeoDataFrame export is available when the optional GIS stack is installed.


In [ ]:
geojson = lum.compare_sites_rankings_geojson(country="GB", sites=candidate_sites.to_dict(orient="records"))
feature_sample = geojson["features"][0] if geojson["features"] else {}

print(f"GeoJSON features: {len(geojson['features'])}")
print(f"Feature sample keys: {list(feature_sample.get('properties', {}).keys())[:8]}")

try:
    gdf = lum.compare_sites_rankings_geodataframe(country="GB", sites=candidate_sites.to_dict(orient="records"))
    display(gdf.head())
except RuntimeError as exc:
    print(f"GeoDataFrame export skipped: {exc}")


## Exercises

- Add a fifth site and compare how the ranking table changes.
- Move the candidate set into another GB region and note where the SSEN headroom helper stops resolving useful DNO context.
- Replace the sample coordinates with a real asset list from your portfolio screen.


## Pitfall

The distribution headroom helper uses public SSEN data only. Keep that caveat visible in the notebook so analysts do not mistake a missing SSEN match for a GB-wide DNO clearance.


## Extension

If a site survives this GIS pass, move into the BESS shortlist notebook to blend siting, revenue, queue, and DNO screening into one ranked storage flow.


In [ ]:
lum.close()
